<a href="https://colab.research.google.com/github/rohaanravi27/E-Commerce-Customer-Segmentation/blob/main/E_Commerce_Customer_Segmentation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

import pandas as pd
import datetime as dt
import numpy as np

np.random.seed(42)
n_rows = 1000

data = {
    'CustomerID': np.random.randint(10000, 11000, size=n_rows),
    'InvoiceNo': np.random.randint(500000, 510000, size=n_rows),
    'Quantity': np.random.randint(1, 10, size=n_rows),
    'UnitPrice': np.random.uniform(2.0, 50.0, size=n_rows),
    'InvoiceDate': [dt.datetime(2026, 1, 1) + dt.timedelta(days=int(np.random.randint(0, 180))) for _ in range(n_rows)]
}

df = pd.DataFrame(data)

df.loc[np.random.choice(df.index, 20, replace=False), 'CustomerID'] = np.nan
df.loc[np.random.choice(df.index, 10, replace=False), 'Quantity'] = -5

print("Local dataset created successfully!")
print(df.head())


Local dataset created successfully!
   CustomerID  InvoiceNo  Quantity  UnitPrice InvoiceDate
0     10102.0     505124         3  20.317373  2026-04-23
1     10435.0     501015         3   8.184738  2026-03-02
2     10860.0     502193         3  21.812534  2026-05-17
3     10270.0     508415         6  40.513008  2026-01-06
4     10106.0     507449         1  24.995363  2026-04-19


In [5]:
import datetime as dt

df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']


snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

print("Data cleaned and RFM metrics calculated successfully!")
print(rfm.head())



Data cleaned and RFM metrics calculated successfully!
            Recency  Frequency    Monetary
CustomerID                                
10000.0          56          1  112.474785
10001.0          56          3  151.090213
10004.0          80          2  486.323563
10007.0         166          1  166.087345
10008.0          88          2  468.431752


In [4]:
import pandas as pd

rfm['R'] = pd.qcut(rfm['Recency'], 5, labels=['5', '4', '3', '2', '1']).astype(str) # Lower recency is better
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=['1', '2', '3', '4', '5']).astype(str)
rfm['M'] = pd.qcut(rfm['Monetary'], 5, labels=['1', '2', '3', '4', '5']).astype(str)

def assign_segment(row):
    r, f, m = row['R'], row['F'], row['M']
    if r in ['4', '5'] and f in ['4', '5']:
        return 'Champions'
    elif r in ['2', '3', '4', '5'] and f in ['3', '4', '5']:
        return 'Loyal Customers'
    elif r in ['1', '2'] and f in ['4', '5'] and m in ['4', '5']:
        return 'Cant Lose Them'
    elif r in ['1', '2'] and f in ['1', '2']:
        return 'Hibernating / Lost'
    else:
        return 'Regular Customers'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

insights = rfm.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'sum', 'count']
}).round(1)

print("\n--- FINAL SEGMENTATION REPORT FOR PLACEMENTS ---")
print(insights)



--- FINAL SEGMENTATION REPORT FOR PLACEMENTS ---
                   Recency Frequency Monetary               
                      mean      mean     mean      sum count
Segment                                                     
Cant Lose Them       141.6       2.1    384.1   2688.8     7
Champions             20.6       2.6    329.5  45135.1   137
Hibernating / Lost   130.6       1.0    137.4  17310.4   126
Loyal Customers       70.0       1.8    237.2  42459.5   179
Regular Customers     66.5       1.0    137.4  21023.1   153
